In [427]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib
import shap
from math import sqrt
from imblearn.over_sampling import SMOTE
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from sklearn import svm
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from matplotlib import font_manager, rc
path = "c:/Windows/Fonts/malgun.ttf"
if platform.system() == 'Darwin':
    rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    font_name = font_manager.FontProperties(fname=path).get_name()
    rc('font', family=font_name)
else:
    print('Unknown system... sorry~~~~')

In [542]:
data = pd.read_excel("C:/Users/PC/Desktop/창원_매출액.xlsx")
data.head()

,id,단지명,업체명,입주일,대업종,중업종,소업종,근로자수,재해자구분,발생형태,...,영업이익_2018,영업이익_2019,영업이익_2020,영업이익_2021,주요제품,벤처여부,이노비즈여부,메인비즈여부,기업부설연구소여부,연구개발전담부서여부
0,19903.0,창원국가산업단지,범진테크,20190827.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,4,사고부상자,부딪힘,...,0.0,58267.0,51595.0,0.0,NaN,부,부,부,부,부
1,8262.0,창원국가산업단지,에이치케이테크,20180405.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,8,사고부상자,물체에 맞음,...,187839.0,274427.0,146554.0,136373.0,환경 및 수처리 설비,여,부,부,부,여
2,26580.0,창원국가산업단지,지성정판,20210512.0,제조업,출판·인쇄·제본또는인쇄물가공업,인쇄업,0,질병이환자,작업관련질병(뇌심 등),...,26362.0,88075.0,43890.0,63306.0,책자 출판 등,부,부,부,부,부
3,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,절단·베임·찔림,...,181128.0,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여
4,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,화학물질누출·접촉,...,181128.0,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여


In [543]:
data.대업종.value_counts()

제조업          1668
기타의사업          33
건설업            26
운수·창고·통신업       1
Name: 대업종, dtype: int64

In [544]:
df = data[["재해발생년도", "매출액_2016", "매출액_2017", "매출액_2018", "매출액_2019", "매출액_2020", "매출액_2021"]]
df.head()

,재해발생년도,매출액_2016,매출액_2017,매출액_2018,매출액_2019,매출액_2020,매출액_2021
0,2019,0,562189,0,742474,740387,0
1,2020,0,450879,1811333,3534300,3029757,4306587
2,2018,0,0,793726,1008817,731050,1042805
3,2018,996121,1428560,1427763,1282601,1174473,1271323
4,2017,996121,1428560,1427763,1282601,1174473,1271323


In [545]:
df.매출액 = 0
for i, j in enumerate(df.재해발생년도.values, 0):
    if j == 2016:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2016"]
    elif j == 2017:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2017"]
    elif j == 2018:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2018"]
    elif j == 2019:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2019"]
    elif j == 2020:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2020"]
    elif j == 2021:
        df.loc[i, "매출액"] = df.loc[i, "매출액_2021"]
    else:
        df.loc[i, "매출액"] = 0


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [546]:
df_매출액 = df[["매출액"]]
df_매출액.head()

,매출액
0,742474.0
1,3029757.0
2,793726.0
3,1427763.0
4,1428560.0


In [547]:
data = pd.concat([data, df_매출액], axis = 1)
data.head()

,id,단지명,업체명,입주일,대업종,중업종,소업종,근로자수,재해자구분,발생형태,...,영업이익_2019,영업이익_2020,영업이익_2021,주요제품,벤처여부,이노비즈여부,메인비즈여부,기업부설연구소여부,연구개발전담부서여부,매출액
0,19903.0,창원국가산업단지,범진테크,20190827.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,4,사고부상자,부딪힘,...,58267.0,51595.0,0.0,NaN,부,부,부,부,부,742474.0
1,8262.0,창원국가산업단지,에이치케이테크,20180405.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,8,사고부상자,물체에 맞음,...,274427.0,146554.0,136373.0,환경 및 수처리 설비,여,부,부,부,여,3029757.0
2,26580.0,창원국가산업단지,지성정판,20210512.0,제조업,출판·인쇄·제본또는인쇄물가공업,인쇄업,0,질병이환자,작업관련질병(뇌심 등),...,88075.0,43890.0,63306.0,책자 출판 등,부,부,부,부,부,793726.0
3,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,절단·베임·찔림,...,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여,1427763.0
4,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,화학물질누출·접촉,...,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여,1428560.0


In [548]:
df1 = data[["대업종", "중업종", "근로자수", "매출액"]]
df1.head()

,대업종,중업종,근로자수,매출액
0,제조업,기계기구·금속·비금속광물제품제조업,4,742474.0
1,제조업,기계기구·금속·비금속광물제품제조업,8,3029757.0
2,제조업,출판·인쇄·제본또는인쇄물가공업,0,793726.0
3,제조업,기계기구·금속·비금속광물제품제조업,8,1427763.0
4,제조업,기계기구·금속·비금속광물제품제조업,8,1428560.0


In [549]:
df1['기업규모'] = 0
df1.loc[df1['근로자수'] >= 5, '기업규모'] = 1
df1.loc[df1['근로자수'] >= 50, '기업규모'] = 2
df1.loc[df1['근로자수'] >= 100, '기업규모'] = 3
df1.loc[df1['근로자수'] >= 300, '기업규모'] = 4
df1.loc[df1['근로자수'] >= 1000, '기업규모'] = 5
df1.drop(columns = ["근로자수"], inplace = True)
df1.head()


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,대업종,중업종,매출액,기업규모
0,제조업,기계기구·금속·비금속광물제품제조업,742474.0,0
1,제조업,기계기구·금속·비금속광물제품제조업,3029757.0,1
2,제조업,출판·인쇄·제본또는인쇄물가공업,793726.0,0
3,제조업,기계기구·금속·비금속광물제품제조업,1427763.0,1
4,제조업,기계기구·금속·비금속광물제품제조업,1428560.0,1


In [553]:
grouped = df2.groupby(by=['대업종', '기업규모'])
df2_grouped = grouped.매출액.median().unstack()
df2_grouped

기업규모,0,1,2,3,4,5
대업종,,,,,,
건설업,1533725.0,NaN,3.708635e+09,2.640053e+09,1.828661e+09,3.708635e+09
기타의사업,557174.0,4848955.0,2.348209e+07,NaN,NaN,NaN
제조업,757522.5,4672979.0,2.550864e+07,6.178596e+07,4.774234e+08,4.101664e+09


In [554]:
df2_0_건설업 = df2_grouped.iloc[0,0]
df2_1_건설업 = df2_grouped.iloc[0,1]
df2_2_건설업 = df2_grouped.iloc[0,2]
df2_3_건설업 = df2_grouped.iloc[0,3]
df2_4_건설업 = df2_grouped.iloc[0,4]
df2_5_건설업 = df2_grouped.iloc[0,5]
df2_0_기타의사업 = df2_grouped.iloc[1,0]
df2_1_기타의사업 = df2_grouped.iloc[1,1]
df2_2_기타의사업 = df2_grouped.iloc[1,2]
df2_3_기타의사업 = df2_grouped.iloc[1,3]
df2_4_기타의사업 = df2_grouped.iloc[1,4]
df2_5_기타의사업 = df2_grouped.iloc[1,5]
df2_0_제조업 = df2_grouped.iloc[2,0]
df2_1_제조업 = df2_grouped.iloc[2,1]
df2_2_제조업 = df2_grouped.iloc[2,2]
df2_3_제조업 = df2_grouped.iloc[2,3]
df2_4_제조업 = df2_grouped.iloc[2,4]
df2_5_제조업 = df2_grouped.iloc[2,5]

In [555]:
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 0)&(df1["매출액"] == 0), '매출액'] = df2_0_제조업
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 1)&(df1["매출액"] == 0), '매출액'] = df2_1_제조업
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 2)&(df1["매출액"] == 0), '매출액'] = df2_2_제조업
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 3)&(df1["매출액"] == 0), '매출액'] = df2_3_제조업
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 4)&(df1["매출액"] == 0), '매출액'] = df2_4_제조업
df1.loc[(df1.대업종 == '제조업')&(df1.기업규모 == 5)&(df1["매출액"] == 0), '매출액'] = df2_5_제조업
df1.loc[(df1.대업종 == '기타의사업')&(df1.기업규모 == 1)&(df1["매출액"] == 0), '매출액'] = df2_1_기타의사업
df1.loc[(df1.대업종 == '기타의사업')&(df1.기업규모 == 4)&(df1["매출액"] == 0), '매출액'] = df2_1_기타의사업
df1.loc[(df1.대업종 == '건설업')&(df1.기업규모 == 5)&(df1["매출액"] == 0), '매출액'] = df2_5_건설업


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [556]:
df1_매출액 = df1.매출액

In [557]:
data.drop(columns = ["매출액"], inplace = True)

In [558]:
data = pd.concat([data, df1_매출액], axis = 1)
data

,id,단지명,업체명,입주일,대업종,중업종,소업종,근로자수,재해자구분,발생형태,...,영업이익_2019,영업이익_2020,영업이익_2021,주요제품,벤처여부,이노비즈여부,메인비즈여부,기업부설연구소여부,연구개발전담부서여부,매출액
0,19903.0,창원국가산업단지,범진테크,20190827.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,4,사고부상자,부딪힘,...,58267.0,51595.0,0.0,NaN,부,부,부,부,부,742474.0
1,8262.0,창원국가산업단지,에이치케이테크,20180405.0,제조업,기계기구·금속·비금속광물제품제조업,각종기계또는동부속품제조업,8,사고부상자,물체에 맞음,...,274427.0,146554.0,136373.0,환경 및 수처리 설비,여,부,부,부,여,3029757.0
2,26580.0,창원국가산업단지,지성정판,20210512.0,제조업,출판·인쇄·제본또는인쇄물가공업,인쇄업,0,질병이환자,작업관련질병(뇌심 등),...,88075.0,43890.0,63306.0,책자 출판 등,부,부,부,부,부,793726.0
3,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,절단·베임·찔림,...,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여,1427763.0
4,19324.0,창원국가산업단지,태성열처리,20140911.0,제조업,기계기구·금속·비금속광물제품제조업,열처리사업,8,사고부상자,화학물질누출·접촉,...,172804.0,143675.0,79575.0,금속열처리도금 외,부,부,부,부,여,1428560.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723,14742.0,창원국가산업단지,다온디자인,20141212.0,제조업,전기기계기구·정밀기구·전자제품제조업,기타전기기계기구제조업,113,사고부상자,끼임,...,NaN,NaN,NaN,NaN,부,부,부,부,부,61785957.0
1724,14742.0,창원국가산업단지,다온디자인,20141212.0,제조업,전기기계기구·정밀기구·전자제품제조업,기타전기기계기구제조업,76,사고부상자,끼임,...,NaN,NaN,NaN,NaN,부,부,부,부,부,25508637.0
1725,14742.0,창원국가산업단지,다온디자인,20141212.0,제조업,전기기계기구·정밀기구·전자제품제조업,기타전기기계기구제조업,90,사고부상자,불균형 및 무리한동작,...,NaN,NaN,NaN,NaN,부,부,부,부,부,25508637.0
1726,5315.0,창원국가산업단지,(주)제이테크,19851113.0,제조업,화학및고무제품제조업,유기화학제품제조업,31,사고부상자,사업장외 교통사고,...,1389624.0,678119.0,0.0,"촉매제, 크롬도금",부,부,부,여,부,12145255.0


In [560]:
data.to_excel("C:/Users/PC/Desktop/창원_매출액2.xlsx", index = False)